# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** (full eval, 1 epoch) | ~60-90 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/anhnvVNU/Day21-Track3-Finetuning-Lab-2A202601579-NgoVietAnh.git"
REPO_DIR = "Day21-Track3-Finetuning-Lab-2A202601579-NgoVietAnh"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/" + REPO_DIR)
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Read HF_TOKEN from Colab Secrets (key icon in the left sidebar).
# The token is never written into this notebook or committed to Git.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: loaded from Colab Secrets")
except Exception:
    print("HF_TOKEN: not configured (optional until Hub upload)")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
EPOCHS       = "1"         # @param ["1", "2", "3"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ["EPOCHS"] = EPOCHS
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


In [ ]:
# @title 5. Download artifacts để hoàn thiện report và nộp bài
import os, shutil
bundle = "/content/lab21_2A202601579_artifacts"
shutil.make_archive(bundle, "zip", root_dir=os.getcwd(),
                    base_dir="results")
# Adapter lớn được đóng gói riêng để tránh phải tải lại khi chỉ sửa report.
if os.path.isdir("adapters/correct"):
    shutil.make_archive(bundle + "_adapter", "zip", root_dir=os.getcwd(),
                        base_dir="adapters/correct")
from google.colab import files
files.download(bundle + ".zip")
if os.path.exists(bundle + "_adapter.zip"):
    files.download(bundle + "_adapter.zip")
